# Scaling study: mesh density, input context, forecast horizon

Three questions, one axis each:

1. **Mesh density** — how much geographic resolution actually helps,
   and where the compute stops paying for itself.
2. **Input context** — does a longer lookback fix the turning problem?
   Failure analysis found turn angle separates the worst-predicted
   windows from the best by a factor of 4, and a longer history is the
   cheapest way to give the model turn *rate* rather than just current
   heading.
3. **Forecast horizon** — how far ahead is worth predicting before the
   forecast stops being useful.

Run the sweeps first (see `slurm/sweep.slurm`), then this notebook reads
the JSON they produce.

**Read these as trends, not as final numbers.** The sweeps use one day
of AIS, a reduced vessel count, and a short epoch budget so that a whole
axis fits in one job. Absolute values will be worse than a full run; the
shape of each curve is the point.

In [1]:
import json, glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110


def load_sweep(out_dir):
    rows = []
    for p in sorted(glob.glob(os.path.join(out_dir, '*.json'))):
        with open(p) as f:
            m = json.load(f)
        if 'error' in m:
            print(f"  skipped {os.path.basename(p)}: {m['error']}")
            continue
        rows.append(m)
    if not rows:
        print(f"  no results in {out_dir}")
        return pd.DataFrame()
    return pd.DataFrame(rows)


def plot_axis(df, x, xlabel, title, logx=False):
    """
    Error and calibration against one swept parameter.

    Constant velocity is drawn on the error panel because it is the bar
    that matters: a model that tracks it but never crosses it has not
    learned anything a straight line does not already provide.
    """
    if df.empty:
        return None
    df = df.sort_values(x)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    ax = axes[0]
    ax.plot(df[x], df['model_error_km'], 'o-', color='tab:blue', label='model')
    ax.plot(df[x], df['const_velocity_error_km'], 's--', color='tab:red',
            label='constant velocity')
    ax.plot(df[x], df['persistence_error_km'], '^:', color='0.6', label='persistence')
    ax.set_xlabel(xlabel); ax.set_ylabel('mean error (km)')
    ax.set_title('Accuracy'); ax.legend(fontsize=8)
    if logx: ax.set_xscale('log')

    ax = axes[1]
    ax.plot(df[x], df['spread_correlation'], 'o-', color='tab:green')
    ax.axhline(0, color='0.7', lw=0.8)
    ax.set_xlabel(xlabel); ax.set_ylabel('spread ↔ displacement correlation')
    ax.set_title('Uncertainty calibration')
    if logx: ax.set_xscale('log')

    ax = axes[2]
    ax.plot(df[x], df['wall_sec']/60, 'o-', color='tab:purple')
    ax.set_xlabel(xlabel); ax.set_ylabel('wall time per config (min)')
    ax.set_title('Cost')
    if logx: ax.set_xscale('log')

    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    return fig

## 1. Mesh density

The mesh is the model's only source of geography — where land is, where
ports are, how a channel narrows. Denser meshes describe the coastline
more finely, but cost grows fast: every extra node is more message
passing on every snapshot, and the graph is rebuilt for every timestamp.

What to look for: the point where the error curve flattens. Beyond it
you are paying compute for resolution the model cannot use — most likely
because vessel displacement per step is already larger than the mesh
spacing, so finer geometry is invisible between timesteps.

In [2]:
mesh_df = load_sweep('results/sweep_mesh')
if not mesh_df.empty:
    display(mesh_df[['n_open_water', 'n_coastal', 'mesh_nodes', 'mesh_edges',
                     'model_error_km', 'const_velocity_error_km',
                     'beats_const_velocity_pct', 'spread_correlation',
                     'wall_sec']].round(3))
    fig = plot_axis(mesh_df, 'mesh_nodes', 'mesh nodes',
                    'Mesh density', logx=True)
    plt.show()

  no results in results/sweep_mesh


### Cross-check against vessel displacement

A mesh finer than the distance a vessel travels per timestep cannot
help: the vessel's nearest mesh nodes barely change between steps, so
the extra resolution never enters the model. Earlier measurement on
real data put median displacement for an underway vessel at ~2.5 km per
15 minutes, against ~6 km open-water mesh spacing at the default
density. That predicts diminishing returns somewhere near the default —
worth checking whether the curve above agrees.

In [ ]:
if not mesh_df.empty:
    # crude spacing estimate: domain area / node count
    domain_km2 = (122 - 99) * 111 * (24 + 3) * 111 * 0.5   # rough, Danish bounds
    mesh_df = mesh_df.sort_values('mesh_nodes').copy()
    mesh_df['approx_spacing_km'] = np.sqrt(domain_km2 / mesh_df['mesh_nodes'])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(mesh_df['approx_spacing_km'], mesh_df['model_error_km'], 'o-')
    ax.axvline(2.5, color='crimson', ls='--',
               label='median vessel displacement / 15 min')
    ax.set_xlabel('approx. mesh spacing (km)')
    ax.set_ylabel('model error (km)')
    ax.set_title('Does mesh resolution below per-step displacement buy anything?')
    ax.legend(fontsize=9)
    plt.show()

## 2. Input context length

Each context step is 15 minutes, so `seq_len` 4 → 24 spans a 1-hour to
6-hour lookback.

The motivating result: turning dominates failures. With only a few
context steps the model sees current heading but little about *turn
rate*, so a vessel mid-manoeuvre looks like one going straight. More
history should help — if the model can use it.

The cost is data, not just compute: longer windows need more consecutive
timesteps per vessel, so both window count and **vessel count** fall.
Vessel diversity matters more for generalization than raw window count,
so watch that column.

In [ ]:
ctx_df = load_sweep('results/sweep_context')
if not ctx_df.empty:
    display(ctx_df[['seq_len', 'lookback_min', 'train_windows', 'train_vessels',
                    'model_error_km', 'const_velocity_error_km',
                    'beats_const_velocity_pct', 'spread_correlation',
                    'wall_sec']].round(3))
    fig = plot_axis(ctx_df, 'lookback_min', 'lookback (minutes)',
                    'Input context length')
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(ctx_df['lookback_min'], ctx_df['train_vessels'], 'o-', color='tab:orange',
            label='vessels')
    ax2 = ax.twinx()
    ax2.plot(ctx_df['lookback_min'], ctx_df['train_windows'], 's--', color='tab:blue',
             label='windows')
    ax.set_xlabel('lookback (minutes)'); ax.set_ylabel('training vessels', color='tab:orange')
    ax2.set_ylabel('training windows', color='tab:blue')
    ax.set_title('Longer context costs data — vessels fall faster than windows')
    plt.show()

## 3. Forecast horizon

`future_len` 2 → 16 is a 30-minute to 4-hour forecast.

Earlier measurement found error grows close to **linearly** with
horizon (~1.9 km per 15-minute step), not exponentially — so longer
forecasts degrade predictably rather than collapsing. The question is
where the model stops beating dead reckoning, since constant-velocity
error also grows with horizon and the two may not diverge.

In [ ]:
hor_df = load_sweep('results/sweep_horizon')
if not hor_df.empty:
    display(hor_df[['future_len', 'horizon_min', 'train_windows',
                    'model_error_km', 'const_velocity_error_km',
                    'beats_const_velocity_pct', 'spread_correlation',
                    'wall_sec']].round(3))
    fig = plot_axis(hor_df, 'horizon_min', 'forecast horizon (minutes)',
                    'Forecast horizon')
    plt.show()

### Skill relative to dead reckoning

Absolute error necessarily grows with horizon, so it says little on its
own. The informative quantity is the **ratio** of model error to
constant-velocity error: below 1.0 the model is adding something, at or
above 1.0 a straight line would have done as well or better.

In [ ]:
if not hor_df.empty:
    d = hor_df.sort_values('horizon_min')
    ratio = d['model_error_km'] / d['const_velocity_error_km']
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(d['horizon_min'], ratio, 'o-', color='tab:blue')
    ax.axhline(1.0, color='crimson', ls='--', label='parity with dead reckoning')
    ax.fill_between(d['horizon_min'], 0, 1, color='tab:green', alpha=0.08)
    ax.set_xlabel('forecast horizon (minutes)')
    ax.set_ylabel('model error / constant-velocity error')
    ax.set_title('Below the line, the model beats a straight line')
    ax.legend(fontsize=9)
    plt.show()

## Reading it all together

The three axes trade against each other, and the sweep is meant to show
where each stops paying:

- **Mesh density** costs compute per snapshot and nothing else. If the
  error curve is flat past the default, take the cheaper mesh and spend
  the compute on data or epochs instead.
- **Input context** costs vessel diversity, which is the thing most
  likely to limit generalization. Only worth extending if it visibly
  improves accuracy or calibration.
- **Forecast horizon** is a product decision as much as a modelling one:
  push it until the model/dead-reckoning ratio approaches 1.0, since
  past that point you are shipping a straight line with extra steps.

One caveat worth repeating: every number here comes from a short run on
one day of data. A configuration that looks flat at this scale may
separate with 12 days and 60 epochs. Use the sweep to *rank* options and
rule out the clearly-bad ones, then confirm the winner at full scale.